<a href="https://colab.research.google.com/github/youssef-mm/FlyRank-ML-Assignment/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/youssef-mm/FlyRank-ML-Assignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### The Data Contract in Plain Words (Five Answers)
1. **What one row means (Unit of Analysis / Grain):** Exactly one pseudonymized content item (`content_id`) belonging to a specific client (`client_id`).
2. **Which table(s) we will use:**
   - *Starter Slice:* `data/raw/content_refresh_anonymized.csv` (pre-aggregated 90-day client/content slice).
   - *Warehouse Scale:* `dim_content` (content metadata) joined with `fact_content_daily_performance` (daily search/engagement time-series partitioned by month) and `dim_clients` (client tracking start dates).
3. **Which time window:**
   - *Feature Window:* Trailing 90-day historical observation window (days -90 to 0).
   - *Outcome / Target Window:* 30-day comparative outcome window (days -30 to 0 vs days -60 to -31) evaluating relative decline. (In the warehouse panel: a prospective 30-day forward window).
4. **What we predict or rank (Target / Proxy):** A binary indicator of search performance decline (`is_declining_label = 1` when `trend_direction == 'down'`), with model probabilities used to generate a prioritized opportunity ranking for content refresh.
5. **One thing deliberately excluded & why:** `trend_pct` and `trend_direction` are **strictly excluded** from features because `is_declining_label` is mathematically calculated from them. Including them injects 100% target leakage.

In [1]:
import os
import pandas as pd

# Load dataset (local path with Colab raw URL fallback)
data_path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    data_path = "../../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    data_path = "https://raw.githubusercontent.com/youssef-mm/FlyRank-ML-Assignment/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)
print(f"Data Contract Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns across {df['client_id'].nunique()} clients.")


Data Contract Loaded: 30,000 rows x 44 columns across 32 clients.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Classification into Four Buckets

| Bucket | Fields | Meaning & Role in Pipeline | Notes / Rationale |
| :--- | :--- | :--- | :--- |
| **Features** | `impressions_90d`, `clicks_90d`, `avg_position`, `ctr`, `days_since_last_update`, `content_age_days`, `word_count`, `search_volume`, `competition`, `freshness_tier` | Observational signals knowable strictly BEFORE the decision moment. | Input signals used by baseline rules and ML models to rank content. |
| **Label / Proxy** | `is_declining_label` (derived from `trend_direction == 'down'`) | The ground-truth outcome being predicted or ranked. | Evaluated strictly as the target; never used as an input feature. |
| **Context** | `content_id`, `client_id`, `content_type`, `main_intent` | Pseudonymous identifiers and categorical grouping metadata. | Used strictly for joins, grouping, client-holdout splits, and reporting. Never learned as numeric features. |
| **Excluded** | `trend_pct`, `trend_direction`, `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `provider_used`, `model_used` | Target components, future/target window metrics, or internal generation metadata. | **Why:** `trend_pct`/`trend_direction` cause direct target leakage. `*_last_30d` columns overlap the target evaluation window. LLM fields have high missingness and no ranking relevance. |

In [2]:
# Verifying Field Classification and Missingness across Buckets
buckets = {
    "Features": ['impressions_90d', 'clicks_90d', 'avg_position', 'days_since_last_update', 'word_count', 'search_volume', 'competition'],
    "Label": ['trend_direction'],
    "Context": ['content_id', 'client_id', 'content_type'],
    "Excluded (Leakage/Future)": ['trend_pct', 'impressions_last_30d', 'clicks_last_30d']
}

print("Field Classification & Missingness Audit:")
for bucket_name, cols in buckets.items():
    print(f"\n[{bucket_name}]:")
    for col in cols:
        null_pct = df[col].isna().mean() * 100
        print(f" - {col:25} | Missing: {null_pct:5.2f}% | Dtype: {df[col].dtype}")


Field Classification & Missingness Audit:

[Features]:
 - impressions_90d           | Missing:  0.00% | Dtype: int64
 - clicks_90d                | Missing:  0.00% | Dtype: int64
 - avg_position              | Missing:  0.00% | Dtype: float64
 - days_since_last_update    | Missing:  0.00% | Dtype: int64
 - word_count                | Missing: 25.66% | Dtype: float64
 - search_volume             | Missing:  8.23% | Dtype: float64
 - competition               | Missing:  8.23% | Dtype: float64

[Label]:
 - trend_direction           | Missing:  0.00% | Dtype: str

[Context]:
 - content_id                | Missing:  0.00% | Dtype: str
 - client_id                 | Missing:  0.00% | Dtype: str
 - content_type              | Missing:  0.00% | Dtype: str

[Excluded (Leakage/Future)]:
 - trend_pct                 | Missing: 11.29% | Dtype: float64
 - impressions_last_30d      | Missing:  0.00% | Dtype: int64
 - clicks_last_30d           | Missing:  0.00% | Dtype: int64


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# -------------------------------------------------------------
# THREE CONTRACT VERIFICATION QUERIES
# -------------------------------------------------------------

# Query 1: Grain Verification (Prove exactly 1 row = 1 unique content_id)
duplicates = df.groupby('content_id').size().reset_index(name='n').query('n > 1')
print("=" * 65)
print("QUERY 1: Grain Verification")
print(f"Duplicate content_ids found: {len(duplicates)}")
assert len(duplicates) == 0, "Grain violation detected!"
print("✔ Fact 1 Verified: Exactly 1 row = 1 unique content_id (30,000 distinct items).\n")

# Query 2: Slice Row Count & Observation Window
print("QUERY 2: Slice Counts and Window Span")
print(f"Total row count: {len(df):,} rows")
print(f"Content age span: Min = {df['content_age_days'].min()} days | Max = {df['content_age_days'].max()} days")
print(f"Days since last update: Min = {df['days_since_last_update'].min()} days | Max = {df['days_since_last_update'].max()} days")
assert df['content_age_days'].min() >= 90, "Window violation: found content with <90 days age!"
print("✔ Fact 2 Verified: All items span >= 90 days, fulfilling trailing window requirements.\n")

# Query 3: Availability Check using IS TRUE filter logic
print("QUERY 3: Data Availability Verification")
# In warehouse Parquet, we filter on three-valued flags: ga4_data_available IS TRUE.
# In this slice, verify the search availability flag and measurable volume opportunity:
has_gsc_search_data = (df['impressions_90d'] > 0) == True
has_ga4_engagement_data = (df['sessions_90d'] > 0) == True
measurable_opportunity = ((df['impressions_90d'] >= 100) & (df['sessions_90d'] > 0)) == True

print(f"Rows with active GSC Search data (IS TRUE): {has_gsc_search_data.sum():,} ({has_gsc_search_data.mean():.1%})")
print(f"Rows with active GA4 Engagement data (IS TRUE): {has_ga4_engagement_data.sum():,} ({has_ga4_engagement_data.mean():.1%})")
print(f"Rows surviving Measurable Opportunity filter (IS TRUE): {measurable_opportunity.sum():,} ({measurable_opportunity.mean():.1%})")
print("✔ Fact 3 Verified: Availability flags checked with boolean filter logic.\n")

# -------------------------------------------------------------
# FIVE-FEATURE FRAME
# -------------------------------------------------------------
print("=" * 65)
print("FIVE-FEATURE FRAME")
honest_feature_cols = ['impressions_90d', 'avg_position', 'days_since_last_update', 'search_volume', 'word_count']
features_df = df[['content_id'] + honest_feature_cols].copy()
print(f"Honest 5-Feature Frame Shape: {features_df.shape} (1 context ID + 5 features)")
print(features_df.head())

# -------------------------------------------------------------
# THE TRAP: DELIBERATE LEAKAGE EXPERIMENT
# -------------------------------------------------------------
print("\n" + "=" * 65)
print("THE TRAP: DELIBERATE LEAKAGE EXPERIMENT")

# Define honest target and features
y = (df['trend_direction'] == 'down').astype(int)
X_honest = features_df[honest_feature_cols].copy()
X_honest['search_volume'] = X_honest['search_volume'].fillna(0)
X_honest['word_count'] = X_honest['word_count'].fillna(X_honest['word_count'].median())

X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)

# 1. Honest Baseline Model
clf_honest = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_honest.fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, clf_honest.predict_proba(X_te)[:, 1])
print(f"Step 1 - Honest Model (5 features) ROC-AUC: {honest_auc:.4f}")

# 2. Injecting The Trap: Add trend_pct (direct mathematical parent of the label)
X_leaked = X_honest.copy()
X_leaked['leaked_trend_pct'] = df['trend_pct'].fillna(0)
X_tr_l, X_te_l, _, _ = train_test_split(X_leaked, y, test_size=0.25, random_state=42, stratify=y)

clf_leaked = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_leaked.fit(X_tr_l, y_tr)
leaked_auc = roc_auc_score(y_te, clf_leaked.predict_proba(X_te_l)[:, 1])
print(f"Step 2 - Leaked Model (5 features + trend_pct) ROC-AUC: {leaked_auc:.4f} (JUMP TOWARD 100%!)")

# 3. Removing The Trap: Restore Honest Feature Set
clf_restored = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_restored.fit(X_tr, y_tr)
restored_auc = roc_auc_score(y_te, clf_restored.predict_proba(X_te)[:, 1])
print(f"Step 3 - Restored Honest Model ROC-AUC: {restored_auc:.4f}")
print("✔ Leakage Trap Experiment Complete: Proved that target-derived features artificially inflate scores.")


QUERY 1: Grain Verification
Duplicate content_ids found: 0
✔ Fact 1 Verified: Exactly 1 row = 1 unique content_id (30,000 distinct items).

QUERY 2: Slice Counts and Window Span
Total row count: 30,000 rows
Content age span: Min = 90 days | Max = 564 days
Days since last update: Min = 1 days | Max = 373 days
✔ Fact 2 Verified: All items span >= 90 days, fulfilling trailing window requirements.

QUERY 3: Data Availability Verification
Rows with active GSC Search data (IS TRUE): 30,000 (100.0%)
Rows with active GA4 Engagement data (IS TRUE): 30,000 (100.0%)
Rows surviving Measurable Opportunity filter (IS TRUE): 22,006 (73.4%)
✔ Fact 3 Verified: Availability flags checked with boolean filter logic.

FIVE-FEATURE FRAME
Honest 5-Feature Frame Shape: (30000, 6) (1 context ID + 5 features)
             content_id  impressions_90d  ...  search_volume  word_count
0  content_304f48230142             3803  ...           10.0      3221.0
1  content_a1fb4e703a9e            15320  ...           90.

### Five Features — Availability at the Decision Moment
Every feature in our five-feature frame is strictly knowable before taking any refresh decision:
1. **`impressions_90d`:** *Knowable at the decision moment because it records GSC search impressions accumulated over the preceding 90-day historical window prior to review.*
2. **`avg_position`:** *Knowable at the decision moment because it reflects the mean ranking position logged by Google Search Console before any refresh decision.*
3. **`days_since_last_update`:** *Knowable at the decision moment because it is computed from the CMS publication or last-modified timestamp available at the moment of inspection.*
4. **`search_volume`:** *Knowable at the decision moment because it is an external keyword search-demand estimate determined before evaluating content performance.*
5. **`word_count`:** *Knowable at the decision moment because it is a static document property extracted from the CMS prior to taking action.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named Limitations of this Data Slice

1. **Unbalanced Panel & Variable Client Tracking Depth:**
   In the warehouse release, client tracking history varies from 3 months to 17 months. Imposing a single global calendar window treats "tracking not yet started" as zero traffic. Per-client observation windows are required.
2. **GSC-Only Early Rows & Three-Valued Flags:**
   Rows before a client's GA4 start date have GA4 columns zero-filled with `ga4_data_available = FALSE` or `NULL`. Filtering with `!= FALSE` silently fails on NULLs; queries must strictly filter using `IS TRUE` to avoid misinterpreting unintegrated analytics as zero user engagement.
3. **Observational Data (No Causal Guarantees):**
   The data records observational search performance, not experimental interventions. While a model can prioritize pages that correlate with decay, it cannot guarantee that updating a page will causally restore search rankings without controlled A/B experiments.
4. **The 3-Day Freshness Cutoff:**
   The warehouse snapshot intentionally excludes the most recent 3 days of data because recent search logs are often incomplete or pending reconciliation. Consequently, intraday volatility or sudden algorithmic shifts from the past 72 hours cannot be detected in this slice.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.